# MedGemma 1.5 4B ? Kvasir-VQA fine-tuning
Train a QLoRA adapter for image + question ? answer using the complete raw Kvasir-VQA split. Run cells in order on a single NVIDIA CUDA GPU. The local RTX 5070 Ti Laptop GPU has 12 GB VRAM. Start with `MAX_STEPS=10`; the forward-pass check below verifies whether the batch fits before training. A larger GPU may be needed depending on sequence length. Training is not executed automatically when this notebook is created.

Sources: [dataset and export format](https://huggingface.co/datasets/SimulaMet-HOST/Kvasir-VQA), [MedGemma model](https://huggingface.co/google/medgemma-1.5-4b-it), [Google fine-tuning example](https://github.com/google-health/medgemma/blob/main/notebooks/fine_tune_with_hugging_face.ipynb).

Accept the model's HAI-DEF terms on Hugging Face before loading it. Kvasir-VQA is CC BY-NC 4.0; cite its authors when reporting results. This notebook produces a research model.

In [4]:
%pip install "transformers==4.57.6" "peft==0.18.1" "accelerate>=1.2,<2" "bitsandbytes>=0.45,<1" "datasets>=3,<6" pandas pillow scikit-learn tqdm sentencepiece protobuf
# Install a CUDA-enabled PyTorch build for your system if torch.cuda.is_available() is False.
# Restart the kernel after installation before continuing.

Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
import json
import random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

DATA_DIR = Path(r"C:\Users\Omen Max\Datasets\Kvasir-VQA") if os.name == 'nt' else Path('/content/datasets/Kvasir-VQA')
OUTPUT_DIR = Path.cwd() / 'outputs' / 'medgemma-1.5-4b-kvasir-vqa'
MODEL_ID = 'google/medgemma-1.5-4b-it'
SEED = 42
EPOCHS = 3
MAX_STEPS = -1  # Set to 10 for a short training smoke run; -1 runs EPOCHS.
RESUME_FROM_CHECKPOINT = None  # Set to a checkpoint directory to resume.
EVAL_GENERATION_SAMPLES = 100  # Set to None for the entire held-out test set.
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('HF_HOME', str(DATA_DIR / '.hf_cache'))
random.seed(SEED)
np.random.seed(SEED)
print('Dataset:', DATA_DIR)
print('Training output:', OUTPUT_DIR)

Dataset: C:\Users\Omen Max\Datasets\Kvasir-VQA
Training output: c:\Users\Omen Max\Marina\Nebras_RandD\VQA\outputs\medgemma-1.5-4b-kvasir-vqa


## Download images and metadata
Uses the requested `load_dataset` and four metadata columns. `drop_duplicates` retains the original row indices reliably across pandas versions and is equivalent to taking the first row per image. Existing valid images are reused. The Hugging Face cache also lives in the dataset folder and requires additional disk space.

In [6]:
"""Export the raw Kvasir-VQA split to metadata.csv and one JPEG per img_id."""
import argparse
import json
import os
from pathlib import Path


def download_dataset(d_path):
    d_path = Path(d_path).expanduser().resolve()
    d_path.mkdir(parents=True, exist_ok=True)
    os.environ.setdefault("HF_HOME", str(d_path / ".hf_cache"))
    from datasets import load_dataset
    from PIL import Image
    from tqdm.auto import tqdm

    ds = load_dataset("SimulaMet-HOST/Kvasir-VQA", cache_dir=str(d_path / ".hf_cache" / "datasets"))
    df = ds['raw'].select_columns(['source', 'question', 'answer', 'img_id']).to_pandas()
    if df.isna().any().any():
        raise ValueError("Dataset contains missing metadata values")
    # Keep original row indices for indexing the raw split, including on older pandas.
    unique = df.drop_duplicates('img_id', keep='first')
    for img_id in unique.img_id:
        if not isinstance(img_id, str) or Path(img_id).name != img_id or any(c in img_id for c in '/\\:'):
            raise ValueError(f"Unsafe image ID: {img_id!r}")
    metadata_tmp = d_path / 'metadata.csv.tmp'
    df.to_csv(metadata_tmp, index=False)
    metadata_tmp.replace(d_path / 'metadata.csv')
    (d_path / 'images').mkdir(exist_ok=True)
    for i, row in tqdm(unique.iterrows(), total=len(unique), desc='Exporting images'):
        destination = d_path / 'images' / f"{row['img_id']}.jpg"
        if destination.exists():
            try:
                with Image.open(destination) as image:
                    image.verify()
                continue
            except (OSError, SyntaxError):
                pass
        temporary = destination.with_suffix('.jpg.tmp')
        ds['raw'][int(i)]['image'].convert('RGB').save(temporary, format='JPEG')
        temporary.replace(destination)
    summary = {'dataset': 'SimulaMet-HOST/Kvasir-VQA', 'split': 'raw',
               'rows': len(df), 'unique_images': len(unique), 'directory': str(d_path)}
    (d_path / 'download_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(json.dumps(summary, indent=2))
    return df



df = download_dataset(DATA_DIR)
display(df.head())

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/30 [00:00<?, ?it/s]

Exporting images:   0%|          | 0/6500 [00:00<?, ?it/s]

{
  "dataset": "SimulaMet-HOST/Kvasir-VQA",
  "split": "raw",
  "rows": 58849,
  "unique_images": 6500,
  "directory": "C:\\Users\\Omen Max\\Datasets\\Kvasir-VQA"
}


,source,question,answer,img_id
0,Ulcerative Colitis,Are there any abnormalities in the image? Chec...,ulcerative colitis,cla820gl0s3nv071u4fgd7xgq
1,Ulcerative Colitis,Are there any anatomical landmarks in the imag...,none,cla820gl0s3nv071u4fgd7xgq
2,Ulcerative Colitis,Are there any instruments in the image? Check ...,none,cla820gl0s3nv071u4fgd7xgq
3,Ulcerative Colitis,Have all polyps been removed?,not relevant,cla820gl0s3nv071u4fgd7xgq
4,Ulcerative Colitis,Is this finding easy to detect?,yes,cla820gl0s3nv071u4fgd7xgq


## Split by image, then construct question?answer examples
All questions for each image stay together in an 80/10/10 train/validation/test split. This prevents shared image IDs across splits, but does not establish patient-level independence because patient identifiers are unavailable. Original answers are preserved; this is open-ended VQA, not binary classification.

In [7]:
from sklearn.model_selection import train_test_split

metadata = pd.read_csv(DATA_DIR / 'metadata.csv', dtype=str, keep_default_na=False)
assert metadata[['img_id', 'question', 'answer']].apply(lambda s: s.str.strip().ne('').all()).all()
image_ids = sorted(metadata.img_id.unique())
train_ids, remaining_ids = train_test_split(image_ids, test_size=0.2, random_state=SEED)
val_ids, test_ids = train_test_split(remaining_ids, test_size=0.5, random_state=SEED)
split_ids = {'train': train_ids, 'validation': val_ids, 'test': test_ids}
assert not (set(train_ids) & set(val_ids) or set(train_ids) & set(test_ids) or set(val_ids) & set(test_ids))
frames = {name: metadata[metadata.img_id.isin(ids)].reset_index(drop=True) for name, ids in split_ids.items()}
for name, frame in frames.items():
    frame.to_csv(OUTPUT_DIR / f'{name}.csv', index=False)
    print(f'{name}: {len(frame):,} questions / {frame.img_id.nunique():,} images')
missing = [i for i in image_ids if not (DATA_DIR / 'images' / f'{i}.jpg').is_file()]
assert not missing, f'Missing images: {missing[:5]}'
(OUTPUT_DIR / 'split_image_ids.json').write_text(json.dumps(split_ids, indent=2))

train: 46,770 questions / 5,200 images
validation: 6,066 questions / 650 images
test: 6,013 questions / 650 images


214557

## Authenticate and load the multimodal model
Use an existing Hugging Face login or `HF_TOKEN`; otherwise the login widget prompts you. Four-bit NF4 weights and LoRA reduce memory use. The vision encoder remains frozen; adapters target language attention projections.

In [9]:
import torch
from huggingface_hub import get_token, notebook_login
from transformers import AutoProcessor, Gemma3ForConditionalGeneration, BitsAndBytesConfig, Trainer, TrainingArguments, set_seed
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if get_token() is None:
    notebook_login()
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU required. Select a GPU notebook kernel / runtime with CUDA-enabled PyTorch.')
set_seed(SEED)
BF16 = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if BF16 else torch.float16
print(torch.cuda.get_device_name(0), COMPUTE_DTYPE)
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = 'right'
model = Gemma3ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=COMPUTE_DTYPE),
    torch_dtype=COMPUTE_DTYPE, device_map={'': 0}, attn_implementation='eager',
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False})
targets = [name for name, _ in model.named_modules()
           if 'language_model' in name and name.rsplit('.', 1)[-1] in {'q_proj', 'k_proj', 'v_proj', 'o_proj'}]
assert targets, 'No language attention targets found; check the model architecture.'
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
    bias='none', task_type='CAUSAL_LM', target_modules=targets))
model.print_trainable_parameters()

NVIDIA GeForce RTX 5070 Ti Laptop GPU torch.bfloat16


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

c:\Users\Omen Max\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Omen Max\Datasets\Kvasir-VQA\.hf_cache\hub\models--google--medgemma-1.5-4b-it. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

trainable params: 8,912,896 || all params: 4,308,992,368 || trainable%: 0.2068


## Multimodal batches with answer-only loss
The collator uses MedGemma's chat template for both prompt and completed conversation. It masks the full prompt, padding, and image tokens. No truncation is applied, so image tokens and target answers are retained. A prefix assertion protects the answer boundary if a future template changes.

In [10]:
class VQADataset(torch.utils.data.Dataset):
    def __init__(self, frame):
        self.rows = frame.to_dict('records')
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        return self.rows[index]

def messages_for(row, include_answer=False):
    messages = [{'role': 'user', 'content': [
        {'type': 'image'}, {'type': 'text', 'text': row['question']}]}]
    if include_answer:
        messages.append({'role': 'assistant', 'content': [{'type': 'text', 'text': row['answer']}]})
    return messages

def collate_fn(rows):
    images = []
    for row in rows:
        with Image.open(DATA_DIR / 'images' / f"{row['img_id']}.jpg") as image:
            images.append([image.convert('RGB')])
    prompts = [processor.apply_chat_template(messages_for(r), tokenize=False, add_generation_prompt=True) for r in rows]
    texts = [processor.apply_chat_template(messages_for(r, True), tokenize=False, add_generation_prompt=False) for r in rows]
    batch = processor(text=texts, images=images, return_tensors='pt', padding=True, truncation=False, add_special_tokens=False)
    prompt_batch = processor(text=prompts, images=images, return_tensors='pt', padding=True, truncation=False, add_special_tokens=False)
    labels = batch['input_ids'].clone()
    for index in range(len(rows)):
        length = int(prompt_batch['attention_mask'][index].sum())
        assert torch.equal(batch['input_ids'][index, :length], prompt_batch['input_ids'][index, :length]), 'Chat prompt is not a prefix'
        labels[index, :length] = -100
    labels[batch['attention_mask'] == 0] = -100
    image_token_id = processor.tokenizer.convert_tokens_to_ids('<image_soft_token>')
    labels[batch['input_ids'] == image_token_id] = -100
    assert (labels != -100).any(dim=1).all(), 'Example has no target tokens'
    batch['labels'] = labels
    return batch

train_data = VQADataset(frames['train'])
val_data = VQADataset(frames['validation'])
sample_batch = collate_fn([train_data[0]])
print({key: tuple(value.shape) for key, value in sample_batch.items()})
print('Supervised tokens:', (sample_batch['labels'] != -100).sum(dim=1).tolist())
with torch.no_grad():
    probe = {key: value.to(model.device) for key, value in sample_batch.items()}
    probe['pixel_values'] = probe['pixel_values'].to(COMPUTE_DTYPE)
    loss = model(**probe).loss
    assert torch.isfinite(loss), 'Non-finite smoke-test loss'
    print('Smoke-test loss:', loss.item())
del sample_batch, probe, loss
torch.cuda.empty_cache()

{'input_ids': (1, 288), 'attention_mask': (1, 288), 'token_type_ids': (1, 288), 'pixel_values': (1, 3, 896, 896), 'labels': (1, 288)}
Supervised tokens: [6]
Smoke-test loss: 6.440748691558838


## Train and save the adapter
Validation runs each epoch; the checkpoint with lowest validation loss is restored. Set `MAX_STEPS=10` for an initial short run. Reduce LoRA rank or use a larger GPU if memory is insufficient. Checkpoints permit resuming interrupted runs.

In [11]:
args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'), num_train_epochs=EPOCHS, max_steps=MAX_STEPS,
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=8, learning_rate=2e-4, warmup_ratio=0.03,
    lr_scheduler_type='cosine', optim='adamw_torch', weight_decay=0.01, max_grad_norm=1.0,
    bf16=BF16, fp16=not BF16, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    eval_strategy='epoch', save_strategy='epoch', logging_steps=10,
    save_total_limit=2, load_best_model_at_end=True, metric_for_best_model='eval_loss',
    greater_is_better=False, prediction_loss_only=True, remove_unused_columns=False,
    dataloader_num_workers=0, dataloader_pin_memory=True, report_to='none', seed=SEED,
)
trainer = Trainer(model=model, args=args, train_dataset=train_data,
    eval_dataset=val_data, data_collator=collate_fn, processing_class=processor)
result = trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)
trainer.save_metrics('train', result.metrics)
trainer.save_metrics('validation', trainer.evaluate())
ADAPTER_DIR = OUTPUT_DIR / 'adapter'
trainer.save_model(str(ADAPTER_DIR))
processor.save_pretrained(ADAPTER_DIR)
import importlib.metadata
config = {'model': MODEL_ID, 'dataset': 'SimulaMet-HOST/Kvasir-VQA', 'seed': SEED,
          'training_args': args.to_dict(), 'packages': {p: importlib.metadata.version(p)
          for p in ['torch', 'transformers', 'peft', 'datasets', 'bitsandbytes']}}
(OUTPUT_DIR / 'run_config.json').write_text(json.dumps(config, indent=2, default=str))
print('Saved LoRA adapter and processor:', ADAPTER_DIR)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## Held-out test generation
Report normalized exact match and word-token F1 on a deterministic test sample, or set the sample limit to `None` for the full test set. These lexical scores can penalize valid paraphrases; inspect the exported predictions as well. The test set is not used for checkpoint selection.

In [ ]:
import re
from collections import Counter

model.eval()
model.config.use_cache = True
model.gradient_checkpointing_disable()

@torch.inference_mode()
def answer_question(image_path, question, max_new_tokens=128):
    with Image.open(image_path) as image:
        image = image.convert('RGB')
    text = processor.apply_chat_template(messages_for({'question': question}), tokenize=False, add_generation_prompt=True)
    inputs = processor(text=text, images=[[image]], return_tensors='pt', add_special_tokens=False).to(model.device)
    inputs['pixel_values'] = inputs['pixel_values'].to(COMPUTE_DTYPE)
    generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.decode(generated[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def normalize(text):
    return ' '.join(re.sub(r'[^\w\s]', ' ', str(text).lower()).split())

def token_f1(prediction, reference):
    p, r = normalize(prediction).split(), normalize(reference).split()
    common = sum((Counter(p) & Counter(r)).values())
    if not p or not r:
        return float(p == r)
    return 2 * common / (len(p) + len(r))

test_frame = frames['test']
if EVAL_GENERATION_SAMPLES is not None:
    test_frame = test_frame.sample(n=min(EVAL_GENERATION_SAMPLES, len(test_frame)), random_state=SEED)
records = []
for row in tqdm(test_frame.to_dict('records'), desc='Test generation'):
    prediction = answer_question(DATA_DIR / 'images' / f"{row['img_id']}.jpg", row['question'])
    records.append({**row, 'prediction': prediction,
        'exact_match': float(normalize(prediction) == normalize(row['answer'])),
        'token_f1': token_f1(prediction, row['answer'])})
predictions = pd.DataFrame(records)
predictions.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)
metrics = {'evaluated_questions': len(predictions), 'total_test_questions': len(frames['test']),
    'exact_match': float(predictions.exact_match.mean()), 'token_f1': float(predictions.token_f1.mean())}
(OUTPUT_DIR / 'test_metrics.json').write_text(json.dumps(metrics, indent=2))
print(metrics)
display(predictions.head(10))

## Reload in a fresh session
Run setup/configuration and authentication first. The saved directory contains a LoRA adapter, not standalone base weights. Reload the original base model with the adapter and use `answer_question` above.

In [ ]:
# Run this cell only in a fresh session, avoiding a second model copy on the GPU.
# from peft import PeftModel
# ADAPTER_DIR = OUTPUT_DIR / 'adapter'
# processor = AutoProcessor.from_pretrained(ADAPTER_DIR)
# base_model = Gemma3ForConditionalGeneration.from_pretrained(
#     MODEL_ID, device_map={'': 0}, torch_dtype=COMPUTE_DTYPE,
#     quantization_config=BitsAndBytesConfig(load_in_4bit=True,
#         bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
#         bnb_4bit_compute_dtype=COMPUTE_DTYPE), attn_implementation='eager')
# model = PeftModel.from_pretrained(base_model, ADAPTER_DIR).eval()
# model.config.use_cache = True